# Data Overview & Analysis

**Complete Data Pipeline Visualization**

Raw Chunks → SFT Candidates → DPO Pairs

Statistics, samples, quality metrics, and distributions.

## Setup

In [1]:
import os
import sys
import json
from pathlib import Path
from collections import Counter, defaultdict

# Initialize project path (portable - works on any computer)
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # Fallback: go up one level if src not in current directory
    PROJECT_ROOT = PROJECT_ROOT.parent
    if not (PROJECT_ROOT / 'src').exists():
        raise FileNotFoundError('Could not find src/ directory. Make sure you run this notebook from the project root.')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from sft import load_chunks
from sft.schema import SFTSample

print(f'✓ Project: {PROJECT_ROOT.name}')

✓ Project: arabic-data-factory-main


In [2]:
# Define paths
CHUNKS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'chunks_asas_albalagha.jsonl'
SFT_CANDIDATES = PROJECT_ROOT / 'data' / 'generated' / 'sft' / 'candidates.jsonl'
SFT_ACCEPTED = PROJECT_ROOT / 'data' / 'generated' / 'sft' / 'accepted.jsonl'
DPO_CANDIDATES = PROJECT_ROOT / 'data' / 'generated' / 'dpo' / 'candidates.jsonl'

print('\n=== DATA FILES ===')
print(f'Raw Chunks:      {CHUNKS_PATH.exists()} ({CHUNKS_PATH})')
print(f'SFT Candidates:  {SFT_CANDIDATES.exists()} ({SFT_CANDIDATES})')
print(f'SFT Accepted:    {SFT_ACCEPTED.exists()} ({SFT_ACCEPTED})')
print(f'DPO Candidates:  {DPO_CANDIDATES.exists()} ({DPO_CANDIDATES})')


=== DATA FILES ===
Raw Chunks:      True (/Users/mohammedalziyad/Desktop/Home/Projects/Tech/arabic-data-factory-main/data/processed/chunks_asas_albalagha.jsonl)
SFT Candidates:  True (/Users/mohammedalziyad/Desktop/Home/Projects/Tech/arabic-data-factory-main/data/generated/sft/candidates.jsonl)
SFT Accepted:    True (/Users/mohammedalziyad/Desktop/Home/Projects/Tech/arabic-data-factory-main/data/generated/sft/accepted.jsonl)
DPO Candidates:  True (/Users/mohammedalziyad/Desktop/Home/Projects/Tech/arabic-data-factory-main/data/generated/dpo/candidates.jsonl)


## 1. Raw Chunks Overview

In [3]:
print('\n=== RAW CHUNKS ANALYSIS ===')

chunks = load_chunks(str(CHUNKS_PATH))
print(f'Total chunks: {len(chunks):,}')

# Analyze chunk structure
regions = Counter(c.get('region') for c in chunks)
formats = Counter(c.get('format_type') for c in chunks)
chunk_sizes = [len(c.get('chunk_text', '')) for c in chunks]

print(f'\nRegions: {dict(regions)}')
print(f'Format types: {dict(formats)}')
print(f'\nChunk text length:')
print(f'  Min: {min(chunk_sizes):,} chars')
print(f'  Max: {max(chunk_sizes):,} chars')
print(f'  Avg: {sum(chunk_sizes)//len(chunk_sizes):,} chars')
print(f'  Total: {sum(chunk_sizes):,} chars')


=== RAW CHUNKS ANALYSIS ===
Total chunks: 394

Regions: {'classical': 394}
Format types: {'dictionary_entry': 394}

Chunk text length:
  Min: 1,767 chars
  Max: 4,644 chars
  Avg: 4,001 chars
  Total: 1,576,463 chars


In [4]:
# Show sample chunks
print('\n=== SAMPLE CHUNKS ===')
for i in range(min(3, len(chunks))):
    chunk = chunks[i]
    print(f'\nChunk {i+1}: {chunk.get("chunk_id")}')
    print(f'  Region: {chunk.get("region")}')
    print(f'  Format: {chunk.get("format_type")}')
    print(f'  Size: {len(chunk.get("chunk_text", ""))} chars')
    print(f'  Text: {chunk.get("chunk_text", "")[:150]}...')


=== SAMPLE CHUNKS ===

Chunk 1: asas_albalagha_c0000
  Region: classical
  Format: dictionary_entry
  Size: 4003 chars
  Text: أبب: اطلب الأمر في إبانه وخذه بربانه * أي أوله وأنشد ابنُ الأعرابي اقد هرمتني قبل إبان الهرم وهي إذا قلت كلي قالت نعم اصحيحة المعدة من كل سقم لو أكلت ...

Chunk 2: asas_albalagha_c0001
  Region: classical
  Format: dictionary_entry
  Size: 4167 chars
  Text: أتى: أتى إليه إحساناً إذا فعله. ووعد الله مأتي. وأتيت الأمر من مأتاه ومأتاته أي من وجهه. قال: وحاجة بت على صماتها أتيتها وحدي من مأتاتها وأتى عليهم ال...

Chunk 3: asas_albalagha_c0002
  Region: classical
  Format: dictionary_entry
  Size: 3986 chars
  Text: أجم: الموت لا تنجو منه الأسد في الآجام والملوك في الآطام وداوم على طعام واحد حتى أجمه أي كرهه.

أحن: تقول: إن الإحن تجُر المحن. وبينهما مضاغنة عظيمة و...


## 2. SFT Candidates Overview

In [5]:
print('\n=== SFT CANDIDATES ANALYSIS ===')

sft_candidates = []
if SFT_CANDIDATES.exists():
    with open(SFT_CANDIDATES, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                sft_candidates.append(data)
    
    print(f'Total SFT candidates: {len(sft_candidates):,}')
    
    # Analyze distributions
    q_types = Counter(s.get('question_type') for s in sft_candidates)
    difficulties = Counter(s.get('difficulty') for s in sft_candidates)
    reasoning_modes = Counter(s.get('reasoning_mode') for s in sft_candidates)
    
    print(f'\nQuestion Types:')
    for qt, count in q_types.most_common():
        print(f'  {qt}: {count:,} ({count/len(sft_candidates)*100:.1f}%)')
    
    print(f'\nDifficulty Levels:')
    for diff, count in difficulties.most_common():
        print(f'  {diff}: {count:,} ({count/len(sft_candidates)*100:.1f}%)')
    
    print(f'\nReasoning Modes (top 5):')
    for mode, count in reasoning_modes.most_common(5):
        print(f'  {mode}: {count:,}')
else:
    print('⚠️  SFT candidates file not found')
    sft_candidates = []


=== SFT CANDIDATES ANALYSIS ===
Total SFT candidates: 4,645

Question Types:
  Definition: 2,107 (45.4%)
  Contextual Meaning: 1,037 (22.3%)
  Meaning: 560 (12.1%)
  Distinction: 297 (6.4%)
  Explanation: 285 (6.1%)
  Cause/Why: 197 (4.2%)
  Comparison: 106 (2.3%)
  Clarification: 28 (0.6%)
  Application: 16 (0.3%)
  Reasoning: 8 (0.2%)
  Relationship: 3 (0.1%)
  Inference: 1 (0.0%)

Difficulty Levels:
  Medium: 2,622 (56.4%)
  Easy: 1,948 (41.9%)
  Hard: 75 (1.6%)

Reasoning Modes (top 5):
  Definition Resolution: 1,770
  Contextual Interpretation: 1,398
  Direct Recall: 1,013
  Comparison: 292
  Disambiguation: 72


In [6]:
# Question and answer length analysis
if sft_candidates:
    q_lengths = [len(s.get('question', '')) for s in sft_candidates]
    a_lengths = [len(s.get('answer', '')) for s in sft_candidates]
    t_lengths = [len(s.get('thinking_arabic', '')) for s in sft_candidates]
    
    print('\n=== SFT TEXT LENGTHS ===')
    print(f'\nQuestion length (chars):')
    print(f'  Min: {min(q_lengths)}, Max: {max(q_lengths)}, Avg: {sum(q_lengths)//len(q_lengths)}')
    
    print(f'\nAnswer length (chars):')
    print(f'  Min: {min(a_lengths)}, Max: {max(a_lengths)}, Avg: {sum(a_lengths)//len(a_lengths)}')
    
    print(f'\nThinking length (chars):')
    print(f'  Min: {min(t_lengths)}, Max: {max(t_lengths)}, Avg: {sum(t_lengths)//len(t_lengths)}')
    
    empty_thinking = sum(1 for t in t_lengths if t == 0)
    print(f'  Empty thinking: {empty_thinking:,} ({empty_thinking/len(sft_candidates)*100:.1f}%)')


=== SFT TEXT LENGTHS ===

Question length (chars):
  Min: 13, Max: 105, Avg: 31

Answer length (chars):
  Min: 13, Max: 283, Avg: 94

Thinking length (chars):
  Min: 0, Max: 372, Avg: 122
  Empty thinking: 764 (16.4%)


In [7]:
# Show sample SFT candidates
if sft_candidates:
    print('\n=== SAMPLE SFT CANDIDATES ===')
    for i in range(min(2, len(sft_candidates))):
        s = sft_candidates[i]
        print(f'\nSample {i+1}: {s.get("sample_id")}')
        print(f'  Type: {s.get("question_type")} | Diff: {s.get("difficulty")}')
        print(f'  Q: {s.get("question")[:80]}...')
        print(f'  A: {s.get("answer")[:80]}...')
        print(f'  T(AR): {s.get("thinking_arabic")[:60]}...' if s.get("thinking_arabic") else '  T(AR): (empty)')
        print(f'  T(EN): {s.get("thinking_english")[:60]}...' if s.get("thinking_english") else '  T(EN): (empty)')


=== SAMPLE SFT CANDIDATES ===

Sample 1: asas_albalagha_c0000_sft_001
  Type: Definition | Diff: Easy
  Q: وش يعني 'أبَّ للمسير'؟...
  A: يعني الشخص تهيأ وتجهز للمشي أو السفر. مثلاً، إذا واحد أبَّ للمسير، يعني صار جاهز...
  T(AR): كلمة 'أبَّ للمسير' في اللغة العربية تعني الاستعداد والتهيؤ ل...
  T(EN): The phrase 'أبَّ للمسير' in Arabic means to prepare and get ...

Sample 2: asas_albalagha_c0000_sft_002
  Type: Comparison | Diff: Medium
  Q: إيش الفرق بين 'أبد الآباد' و 'أبد الأبيد'؟...
  A: ما فيه فرق كبير بينهم، كلها تعني الزمن الطويل جداً أو الأبدية. يعني كلها تدل على...
  T(AR): السؤال يطلب الفرق بين تعبيرين. التعبيرين 'أبد الآباد' و 'أبد...
  T(EN): The question asks for the difference between two expressions...


## 3. SFT Accepted Overview

In [8]:
print('\n=== SFT ACCEPTED ANALYSIS ===')

sft_accepted = []
if SFT_ACCEPTED.exists():
    with open(SFT_ACCEPTED, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                sft_accepted.append(data)
    
    print(f'Total accepted: {len(sft_accepted):,}')
    if sft_candidates:
        accept_rate = len(sft_accepted) / len(sft_candidates) * 100
        print(f'Acceptance rate: {accept_rate:.1f}%')
    
    # Same analysis as candidates
    q_types_acc = Counter(s.get('question_type') for s in sft_accepted)
    difficulties_acc = Counter(s.get('difficulty') for s in sft_accepted)
    
    print(f'\nQuestion Types (Accepted):')
    for qt, count in q_types_acc.most_common():
        print(f'  {qt}: {count:,} ({count/len(sft_accepted)*100:.1f}%)')
    
    print(f'\nDifficulty Levels (Accepted):')
    for diff, count in difficulties_acc.most_common():
        print(f'  {diff}: {count:,} ({count/len(sft_accepted)*100:.1f}%)')
else:
    print('⚠️  SFT accepted file not found')
    sft_accepted = []


=== SFT ACCEPTED ANALYSIS ===
Total accepted: 4,645
Acceptance rate: 100.0%

Question Types (Accepted):
  Definition: 2,107 (45.4%)
  Contextual Meaning: 1,037 (22.3%)
  Meaning: 560 (12.1%)
  Distinction: 297 (6.4%)
  Explanation: 285 (6.1%)
  Cause/Why: 197 (4.2%)
  Comparison: 106 (2.3%)
  Clarification: 28 (0.6%)
  Application: 16 (0.3%)
  Reasoning: 8 (0.2%)
  Relationship: 3 (0.1%)
  Inference: 1 (0.0%)

Difficulty Levels (Accepted):
  Medium: 2,622 (56.4%)
  Easy: 1,948 (41.9%)
  Hard: 75 (1.6%)


## 4. DPO Pairs Overview

In [9]:
print('\n=== DPO CANDIDATES ANALYSIS ===')

dpo_candidates = []
if DPO_CANDIDATES.exists():
    with open(DPO_CANDIDATES, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                dpo_candidates.append(data)
    
    print(f'Total DPO pairs: {len(dpo_candidates):,}')
    if sft_accepted:
        dpo_rate = len(dpo_candidates) / len(sft_accepted) * 100
        print(f'DPO generation rate: {dpo_rate:.1f}%')
    
    # Analyze rejection types
    rejection_types = Counter(p.get('rejection_type') for p in dpo_candidates)
    
    print(f'\nRejection Types:')
    for rej_type, count in rejection_types.most_common():
        print(f'  {rej_type}: {count:,} ({count/len(dpo_candidates)*100:.1f}%)')
else:
    print('⚠️  DPO candidates file not found')
    dpo_candidates = []


=== DPO CANDIDATES ANALYSIS ===
Total DPO pairs: 3,000
DPO generation rate: 64.6%

Rejection Types:
  partial_factual_errors: 3,000 (100.0%)


In [10]:
# Show sample DPO pairs
if dpo_candidates:
    print('\n=== SAMPLE DPO PAIRS ===')
    for i in range(min(2, len(dpo_candidates))):
        p = dpo_candidates[i]
        print(f'\nPair {i+1}: {p.get("pair_id")}')
        print(f'  Rejection type: {p.get("rejection_type")}')
        
        prompt = p.get('prompt', [{}])[0].get('content', '')
        chosen = p.get('chosen', [{}])[0].get('content', '')
        rejected = p.get('rejected', [{}])[0].get('content', '')
        
        print(f'  Prompt: {prompt[:60]}...')
        print(f'  Chosen: {chosen[:60]}...')
        print(f'  Rejected: {rejected[:60]}...')


=== SAMPLE DPO PAIRS ===

Pair 1: asas_albalagha_c0000_sft_001_dpo_00
  Rejection type: partial_factual_errors
  Prompt: وش يعني 'أبَّ للمسير'؟...
  Chosen: Thinking:
كلمة 'أبَّ للمسير' في اللغة العربية تعني الاستعداد...
  Rejected: أبَّ للمسير يعني الشخص صار تعبان ومرهق من كثرة المشي أو السف...

Pair 2: asas_albalagha_c0000_sft_002_dpo_00
  Rejection type: partial_factual_errors
  Prompt: إيش الفرق بين 'أبد الآباد' و 'أبد الأبيد'؟...
  Chosen: Thinking:
السؤال يطلب الفرق بين تعبيرين. التعبيرين 'أبد الآب...
  Rejected: والله يا خوي، 'أبد الآباد' هذي تستخدم للزمن اللي ما له نهاية...


## 5. Pipeline Summary

In [11]:
print('\n' + '='*60)
print('DATA PIPELINE SUMMARY')
print('='*60)
print(f'Raw Chunks:       {len(chunks):>10,}')
print(f'SFT Candidates:   {len(sft_candidates):>10,}')
if sft_candidates:
    print(f'  → {len(sft_candidates)/len(chunks):.1f} samples per chunk')
print(f'SFT Accepted:     {len(sft_accepted):>10,}')
if sft_candidates:
    print(f'  → {len(sft_accepted)/len(sft_candidates)*100:.1f}% acceptance rate')
print(f'DPO Pairs:        {len(dpo_candidates):>10,}')
if sft_accepted:
    print(f'  → {len(dpo_candidates)/len(sft_accepted)*100:.1f}% conversion from SFT')
print('='*60)


DATA PIPELINE SUMMARY
Raw Chunks:              394
SFT Candidates:        4,645
  → 11.8 samples per chunk
SFT Accepted:          4,645
  → 100.0% acceptance rate
DPO Pairs:             3,000
  → 64.6% conversion from SFT


## 6. Quality Insights

In [12]:
print('\n=== KEY METRICS ===')

if sft_candidates and sft_accepted:
    print(f'\n✓ Generation Success:')
    print(f'  {len(sft_candidates):,} candidates from {len(chunks):,} chunks')
    print(f'  {len(sft_candidates)/len(chunks):.1f}x expansion rate')
    
    print(f'\n✓ Quality Filtering:')
    print(f'  {len(sft_accepted):,} passed confidence threshold')
    print(f'  {len(sft_candidates)-len(sft_accepted):,} filtered out')
    print(f'  {len(sft_accepted)/len(sft_candidates)*100:.1f}% pass rate')
    
    if dpo_candidates:
        print(f'\n✓ DPO Pair Generation:')
        print(f'  {len(dpo_candidates):,} preference pairs created')
        print(f'  {len(dpo_candidates)/len(sft_accepted)*100:.1f}% conversion rate')
        
        rejection_types = Counter(p.get('rejection_type') for p in dpo_candidates)
        print(f'  Top rejection type: {rejection_types.most_common(1)[0][0]}')

print('\n✓ All data loaded successfully!')


=== KEY METRICS ===

✓ Generation Success:
  4,645 candidates from 394 chunks
  11.8x expansion rate

✓ Quality Filtering:
  4,645 passed confidence threshold
  0 filtered out
  100.0% pass rate

✓ DPO Pair Generation:
  3,000 preference pairs created
  64.6% conversion rate
  Top rejection type: partial_factual_errors

✓ All data loaded successfully!
